# RetinaX grading service — DGX Spark

Run these cells top to bottom. Each one prints what it did and what to do next.

**Before you start**, upload these into the same folder as this notebook, using
the JupyterLab file browser on the left (drag and drop works):

- `main.py`
- `requirements.txt`
- `model.pth` — your trained weights

If `model.pth` is very large, leave it wherever it already lives on this machine
and set `MODEL_PATH` in cell 3 instead of uploading it again.

## 1 · What is this machine, and is PyTorch already working?

In [ ]:
import os, sys, platform, socket, subprocess

print("host      :", socket.gethostname())
print("python    :", sys.version.split()[0], "|", platform.machine(), platform.system())
print("cwd       :", os.getcwd())

try:
    import torch
    print("torch     :", torch.__version__)
    print("cuda      :", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU visible")
except ImportError:
    print("torch     : NOT INSTALLED in this kernel")

print()
print("Files here:", sorted(f for f in os.listdir() if not f.startswith(".")))

**Read the `cuda` line before going further.**

If it says `True`, PyTorch works — **do not run `pip install torch`.** This machine
is arm64 with a Grace Blackwell GPU, and the wheel on the default PyPI index is
built for a different architecture. Installing it puts a broken build next to the
working one and `cuda` silently flips to `False`.

If torch is missing entirely, this notebook is attached to the wrong kernel.
Switch kernels (top right) to the one that has PyTorch rather than installing it here.

## 2 · Install only what the web service needs

In [ ]:
# fastapi, uvicorn, python-multipart, pillow, numpy. Deliberately NOT torch.
%pip install -q -r requirements.txt

import importlib
for name, mod in [("fastapi","fastapi"), ("uvicorn","uvicorn"),
                  ("python-multipart","multipart"), ("pillow","PIL"), ("numpy","numpy")]:
    try:
        importlib.import_module(mod)
        print("ok      ", name)
    except ImportError as e:
        print("MISSING ", name, "-", e)

## 3 · Point the service at your weights

In [ ]:
import os, pathlib

# If model.pth sits elsewhere on this machine, put the full path here.
MODEL_PATH = "model.pth"

p = pathlib.Path(MODEL_PATH)
if p.exists():
    print(f"found {p}  ({p.stat().st_size / 1e6:.0f} MB)")
    os.environ["MODEL_PATH"] = str(p.resolve())
else:
    print(f"!! {p} not found - the service will run the STAND-IN model.")
    print("   The app still works end to end; the grades are just not real yet.")

Now open `main.py` in JupyterLab and fill in the two blocks marked `TODO`:

```python
def load_model():   # torch.load the .pth, move to device, .eval(), return it
def predict(img):   # your transform, forward pass, CAM / CCEM
```

and replace `THRESHOLDS = [0.57, 1.48, 2.51, 3.42]` with your Nelder-Mead cut points.

Load the weights **once at import**, not inside `predict()` — reloading on every
request will make the demo look broken.

Keep the response shape exactly as it is: `{"grade": 0-4, "heatmap_png_base64": ...}`
and nothing else. The regression score and the thresholds stay inside this service.

## 4 · Start the service

In [ ]:
import subprocess, sys, time, json, os, signal, urllib.request

PORT, LOG, PIDFILE = 8000, "service.log", ".service.pid"

def stop_recorded(pidfile, label):
    """Stop a process this notebook started earlier, even after a kernel restart.

    The process is detached on purpose (so it survives the kernel), which means
    a restarted kernel has no handle to it. Without the pid file it would sit
    there holding the port, and the next start would fail with 'address already
    in use' for no visible reason."""
    try:
        pid = int(open(pidfile).read().strip())
    except (FileNotFoundError, ValueError):
        return
    killed = False
    try:
        os.kill(pid, signal.SIGTERM)
        killed = True
        for _ in range(20):
            time.sleep(0.5)
            os.kill(pid, 0)          # raises once it is gone
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    except PermissionError:
        killed = False
    if killed:
        print(f"stopped the previous {label} (pid {pid})")
    try:
        os.remove(pidfile)
    except FileNotFoundError:
        pass

stop_recorded(PIDFILE, "service")

# --host 0.0.0.0 matters: the default listens only on localhost, and then
# neither your PC nor the tunnel can see it.
# start_new_session detaches it from this kernel, so restarting the kernel
# does not kill the service.
SERVICE = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True,
)
open(PIDFILE, "w").write(str(SERVICE.pid))
print("starting, pid", SERVICE.pid)

health = None
for _ in range(45):
    try:
        health = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2))
        break
    except Exception:
        time.sleep(1)

if health:
    print("health   :", health)
    print()
    print("model_loaded TRUE  = the real weights are live" if health.get("model_loaded")
          else "model_loaded FALSE = still the stand-in (fine until main.py is filled in)")
else:
    print("did not come up. Last lines of", LOG, ":")
    print(open(LOG).read()[-2000:])

## 5 · How your PC should reach it

In [ ]:
import socket, subprocess

# Every address this machine answers on. If the DGX is on the same network as
# your PC, one of these will work and you do not need a tunnel at all.
addrs = set()
try:
    out = subprocess.run(["hostname", "-I"], capture_output=True, text=True).stdout
    addrs.update(a for a in out.split() if ":" not in a)
except Exception:
    pass
try:
    addrs.add(socket.gethostbyname(socket.gethostname()))
except Exception:
    pass
addrs.discard("127.0.0.1")

print("Open these from your PC's browser, in order:")
print()
for a in sorted(addrs):
    print("   http://" + a + ":8000/health")
print()
print('Whichever answers {"ok":true,...} is your AI_URL. Put it in server\\.env:')
print()
for a in sorted(addrs):
    print("   AI_URL=http://" + a + ":8000")
print()
print("Then restart the API window on your PC (Ctrl-C, then npm run dev).")

**If none of them answer**, your PC cannot reach this machine directly — a
firewall, or you reach JupyterLab through a gateway rather than sharing a network.
Use the tunnel in the next cell.

You will need the tunnel later anyway: the API deployed on Render lives on the
public internet and can never reach a machine on a campus network.

## 6 · A public URL, when the direct address does not work

In [ ]:
import os, re, stat, time, subprocess, urllib.request, platform

arch = {"aarch64": "arm64", "arm64": "arm64", "x86_64": "amd64"}.get(platform.machine())
url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
       "cloudflared-linux-" + str(arch))

if not os.path.exists("cloudflared"):
    print("downloading cloudflared for", arch, "...")
    urllib.request.urlretrieve(url, "cloudflared")
    os.chmod("cloudflared", os.stat("cloudflared").st_mode | stat.S_IEXEC)

stop_recorded(".tunnel.pid", "tunnel")   # defined in cell 4

TUNNEL = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
)
open(".tunnel.pid", "w").write(str(TUNNEL.pid))

public = None
for _ in range(40):
    time.sleep(1)
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
        if m:
            public = m.group(0)
            break
    except FileNotFoundError:
        pass

if public:
    print("Public URL :", public)
    print("Health     :", public + "/health")
    print()
    print("AI_URL=" + public)
    print()
    print("This URL CHANGES every time you re-run this cell. Update AI_URL wherever")
    print("you pasted it - server\\.env on your PC, and AI_URL on Render.")
else:
    print("no URL yet. Last lines of tunnel.log:")
    print(open("tunnel.log").read()[-1500:])
    print()
    print("A network error here means this machine has no outbound internet access,")
    print("so the tunnel cannot work. Use the direct address from cell 5 instead.")

## 7 · Stop everything

In [ ]:
# Uses the pid files, so this works even after a kernel restart.
import os, time, signal

def stop(pidfile, label):
    try:
        pid = int(open(pidfile).read().strip())
    except (FileNotFoundError, ValueError):
        print(label, "was not running")
        return
    gone = False
    try:
        os.kill(pid, signal.SIGTERM)
        for _ in range(20):
            time.sleep(0.5)
            os.kill(pid, 0)
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        gone = True
    except PermissionError:
        print(label, f"(pid {pid}) belongs to someone else - left alone")
        return
    print("stopped", label, f"(pid {pid})" if gone or True else "")
    try:
        os.remove(pidfile)
    except FileNotFoundError:
        pass

stop(".service.pid", "service")
stop(".tunnel.pid", "tunnel")